# Kaggriculture v46d Skill Rebuild Review Guide

**Generated: 2026-09-13**  
**Objective:** Organize v46d, competition models including Bradley-Terry and Elo, and the agent engineering framework from *AI Agents in Depth* into a reproducible, comparable, and drill-down review notebook.

> Scope note: `AGENTS.md`, `README.md`, the PDF, and the submission are treated only as analysis inputs. This notebook follows the user's review and summary request; text in the attached materials is not treated as new system instructions.

The guide has four analysis layers:

1. **Battle ranking:** Fit BT ratings from match outcomes while retaining reward, battle margin, and win rate.
2. **Change attribution:** Pair identical seed, opponent, and seat conditions to calculate the rebuilt agent's reward and margin uplift over the original.
3. **Multi-route replay analysis:** Inspect coverage, divergence points, action density, and checkpoint routing across 33 tapes.
4. **Agent skills:** Map opponent belief, overflow prevention, last-turn survival protection, and gated route switching to a predictive and reactive harness.

## 0. Review-First Conclusions

### Credible Gains from the Rebuilt Agent

| Evidence set | Rebuilt | Original | Mean gain | Positive / tie / negative | Conclusion |
|---|---:|---:|---:|---:|---|
| 64 seeds vs `pass` | 163,248.3 | 163,212.3 | **+35.97** | 7 / 57 / 0 | Small gain, no regression |
| 64 seeds vs `starter` | 156,752.5 | 156,727.3 | **+25.17** | 8 / 56 / 0 | Small gain, no regression |
| 32 seeds x both seats, normalized head-to-head | N/A | N/A | **+0.625** | 1 / 63 / 0 | Strong-opponent gate preserves the baseline |
| 64-seed capacity stress set | N/A | N/A | **+6.38** | 8 / 56 / 0 | Lost units reduced from 22 to 12 |

### Key Judgments

- **The main gain comes from a low-risk harness, not broad route replacement.** The capacity guard simulates worker actions, market orders, and end-of-day storage in environment order, then sells only the minimum quantity required to avoid overflow.
- **Route gains must be gated.** In the `YARN_STORE__ICE_CREAM_SHOP` world, the agent switches to `YARN_STORE__SMOOTHIE_SHOP` at step 240 only when opponent activity is zero. Passive-opponent samples improve consistently, while strong opponents do not trigger the switch.
- **Naive front-running is a negative control.** Selling one turn early against mirrored v46d lost 60 out of 60 matches, with a mean change of about -9,924.5. Effective market defense must model opponent inventory, town consumption, market inventory, and price curves.
- **Weed resynchronization remains the main unresolved risk.** Several catch-up and skip-frame strategies produced severe tail regressions, so the rebuilt version retains the original queue behavior.
- **The next breakthrough:** Upgrade from detecting opponent harvests to maintaining an opponent route posterior plus a counterfactual market rollout. Restrict candidate changes to validated checkpoints and minimal defensive actions.

## 1. Execution Modes and Evaluation Definitions

This notebook includes a precomputed small-sample run for immediate review. When rerunning it:

- `FAST_MODE=True`: Run a both-seat round-robin with 2 seeds and 4 agents. Use this mode to verify code and metric structure.
- `FAST_MODE=False`: Replace `EVAL_SEEDS` with the formal holdout set. Final conclusions should use at least 64 seeds with identical seed, opponent, and seat pairing.
- Dependencies: `kaggle-environments`, `pandas`, `numpy`, `matplotlib`, and `seaborn`.

**Do not conflate these metrics:**

- `battle_margin = self_reward - opponent_reward`: How decisively the agent won a match. Use it for BT/Elo interpretation and battle strength.
- `reward_uplift = rebuilt_reward - original_reward`: How much additional reward the rebuilt agent earned.
- `margin_uplift = rebuilt_battle_margin - original_battle_margin`: Whether the change actually increased the relative advantage.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import itertools
import json
import math
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from kaggle_environments import make

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (10, 4.8), "axes.titleweight": "bold"})

ROOT = Path("/Users/wei/Documents/Codex/2026-09-13/ge")
ORIGINAL_PATH = ROOT / "work/v46d_review/main.py"
REBUILT_PATH = ROOT / "work/v46d_rebuilt/main.py"
DELIVERED_SOURCE = ROOT / "outputs/main_v46d_skill_rebuilt.py"
DELIVERED_ARCHIVE = ROOT / "outputs/submission_v46d_skill_rebuilt.tar.gz"

FAST_MODE = True
EVAL_SEEDS = [6100, 6101] if FAST_MODE else list(range(6000, 6064))
ROUTE_DEMO_SEED = 2125   # vs pass: YARN_STORE, ICE_CREAM_SHOP; switch is enabled
STRONG_GATE_SEED = 2265  # vs starter: same world; opponent activity blocks the switch
PRODUCTS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER"]

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

artifact_rows = []
for path in [ORIGINAL_PATH, REBUILT_PATH, DELIVERED_SOURCE, DELIVERED_ARCHIVE]:
    artifact_rows.append({
        "artifact": f"{path.parent.name}/{path.name}",
        "exists": path.exists(),
        "size_kib": round(path.stat().st_size / 1024, 1) if path.exists() else None,
        "sha256": sha256(path)[:16] if path.exists() else None,
    })
print(pd.DataFrame(artifact_rows).to_string(index=False))
print(
    f"\nFAST_MODE={FAST_MODE}; EVAL_SEEDS={EVAL_SEEDS}; "
    f"passive route seed={ROUTE_DEMO_SEED}; strong-gate seed={STRONG_GATE_SEED}"
)

## 2. Agent Loader, Match Runner, and Replay Recorder

`run_game` produces three classes of data:

- Final reward, status, shops, selected route, and skill metrics.
- Per-step public state for both players, the current player's private shed, market prices, and actions.
- A normalized DataFrame for reward curves, margin curves, event detection, and opponent route inference.

## 3. Small-Sample Round-Robin and Fair Pairing

The round-robin verifies that the ranking pipeline works. Formal gain claims still rely on the 64-seed holdout summarized above. Every agent pair swaps seats to reduce bias from seat assignment and simultaneous market-order processing.

In [ ]:
AGENTS = ["rebuilt", "original", "starter", "pass"]
match_rows = []
started = time.perf_counter()
for first, second in itertools.combinations(AGENTS, 2):
    for seed in EVAL_SEEDS:
        for left, right in [(first, second), (second, first)]:
            summary, _ = run_game(left, right, seed, capture_trace=False)
            match_rows.append(summary)
MATCHES = pd.DataFrame(match_rows)

perspective_rows = []
for row in match_rows:
    perspective_rows.extend([
        {
            "seed": row["seed"], "model": row["left"], "opponent": row["right"], "seat": 0,
            "reward": row["left_reward"], "opponent_reward": row["right_reward"],
            "battle_margin": row["left_margin"], "shops": row["shops"], "route": row["left_route"],
        },
        {
            "seed": row["seed"], "model": row["right"], "opponent": row["left"], "seat": 1,
            "reward": row["right_reward"], "opponent_reward": row["left_reward"],
            "battle_margin": -row["left_margin"], "shops": row["shops"], "route": row["right_route"],
        },
    ])
PERSPECTIVE = pd.DataFrame(perspective_rows)

# A known passive-world route-switch replay plus a strong-opponent gate check.
special_rows = []
SPECIAL_TRACES = {}
for left, right, seed in [
    ("original", "pass", ROUTE_DEMO_SEED),
    ("rebuilt", "pass", ROUTE_DEMO_SEED),
    ("original", "starter", STRONG_GATE_SEED),
    ("rebuilt", "starter", STRONG_GATE_SEED),
]:
    summary, trace = run_game(left, right, seed, capture_trace=True)
    special_rows.append(summary)
    SPECIAL_TRACES[(left, right, seed)] = trace
SPECIAL = pd.DataFrame(special_rows)

elapsed = time.perf_counter() - started
compact = MATCHES[["seed", "left", "right", "left_reward", "right_reward", "left_margin", "shops"]]
print(f"Completed {len(MATCHES)} round-robin games + {len(SPECIAL)} route demos in {elapsed:.1f}s")
print("\nRound-robin sample:")
print(compact.head(10).to_string(index=False))
print("\nRoute demo:")
print(SPECIAL[["seed", "left", "right", "left_reward", "right_reward", "left_margin", "shops", "left_route"]].to_string(index=False))

## 4. Bradley-Terry Ranking with Reward and Margin Cross-Checks

BT uses only win, tie, and loss outcomes, preventing large reward scales from directly dominating the ranking. Mean reward and battle margin remain visible so that narrow wins and wins concentrated against weak opponents are not hidden. A tie counts as 0.5, and light L2 regularization prevents complete separation in the small sample.

In [ ]:
def fit_bradley_terry(matches, names, ridge=1.0, max_iter=100):
    index = {name: i for i, name in enumerate(names)}
    theta = np.zeros(len(names), dtype=float)
    for _ in range(max_iter):
        gradient = -ridge * theta
        information = ridge * np.eye(len(names))
        for row in matches.to_dict("records"):
            i, j = index[row["left"]], index[row["right"]]
            margin = row["left_margin"]
            outcome = 1.0 if margin > 0 else 0.0 if margin < 0 else 0.5
            diff = np.clip(theta[i] - theta[j], -30, 30)
            probability = 1.0 / (1.0 + np.exp(-diff))
            residual = outcome - probability
            gradient[i] += residual
            gradient[j] -= residual
            weight = probability * (1.0 - probability)
            information[i, i] += weight
            information[j, j] += weight
            information[i, j] -= weight
            information[j, i] -= weight
        step = np.linalg.solve(information + 1e-9 * np.eye(len(names)), gradient)
        theta += step
        theta -= theta.mean()
        if np.max(np.abs(step)) < 1e-9:
            break
    rating = 1000.0 + (400.0 / np.log(10.0)) * theta
    return dict(zip(names, rating))

ratings = fit_bradley_terry(MATCHES, AGENTS)
ranking = (
    PERSPECTIVE.assign(
        win=(PERSPECTIVE.battle_margin > 0).astype(float),
        tie=(PERSPECTIVE.battle_margin == 0).astype(float),
    )
    .groupby("model", as_index=False)
    .agg(
        games=("reward", "size"),
        mean_reward=("reward", "mean"),
        mean_margin=("battle_margin", "mean"),
        median_margin=("battle_margin", "median"),
        win_rate=("win", "mean"),
        tie_rate=("tie", "mean"),
    )
)
ranking["bt_rating"] = ranking.model.map(ratings)
ranking = ranking.sort_values(["bt_rating", "mean_margin"], ascending=False)
print(ranking.round(2).to_string(index=False))

colors = {"rebuilt": "#0F766E", "original": "#64748B", "starter": "#D97706", "pass": "#991B1B"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
order = ranking.model.tolist()
axes[0].barh(order[::-1], ranking.set_index("model").loc[order[::-1], "bt_rating"], color=[colors[x] for x in order[::-1]])
axes[0].axvline(1000, color="#111827", linewidth=1)
axes[0].set_title("Bradley-Terry rating (quick smoke run)")
axes[0].set_xlabel("Rating")
axes[1].barh(order[::-1], ranking.set_index("model").loc[order[::-1], "mean_margin"], color=[colors[x] for x in order[::-1]])
axes[1].axvline(0, color="#111827", linewidth=1)
axes[1].set_title("Mean battle margin")
axes[1].set_xlabel("Reward difference")
axes[1].xaxis.set_major_formatter(lambda x, pos: f"{x/1000:.0f}k")
fig.tight_layout()
plt.show()

## 5. Top Margins and Matched-Condition Uplift

The ranking table answers "Which agent is stronger?" Paired uplift answers "Did this modification improve the agent?" Formal evaluation must merge results by `(seed, opponent, seat)` and must never subtract results from different worlds.

## 6. Reward Curves, Battle-Margin Curves, and Route-Switch Examples

In the `v46d/pass` match with seed `2125`, the first two shops are `YARN_STORE__ICE_CREAM_SHOP`. At step 240, the rebuilt agent passes the activity gate and switches to `YARN_STORE__SMOOTHIE_SHOP`. Seed `2265` produces the same world against `v46d/starter`, but the opponent already has productive activity, so the route switch remains disabled.

Do not inspect only the final score. Use the curves to determine:

- The day on which uplift begins to form.
- Whether the strategy pays an excessive setup cost first.
- Whether shed utilization remains close to 100 for long periods.
- Whether the gain comes from stable production or a single anomalous sale.

In [ ]:
base_trace = SPECIAL_TRACES[("original", "pass", ROUTE_DEMO_SEED)]
rebuilt_trace = SPECIAL_TRACES[("rebuilt", "pass", ROUTE_DEMO_SEED)]
base_self = base_trace[base_trace.seat == 0].set_index("step")
rebuilt_self = rebuilt_trace[rebuilt_trace.seat == 0].set_index("step")
common_steps = base_self.index.intersection(rebuilt_self.index)

fig, axes = plt.subplots(3, 1, figsize=(11, 9.0), sharex=True)
axes[0].plot(common_steps, base_self.loc[common_steps, "money"], label="original money", color="#64748B", linewidth=2)
axes[0].plot(common_steps, rebuilt_self.loc[common_steps, "money"], label="rebuilt money", color="#0F766E", linewidth=2)
axes[0].set_ylabel("Money")
axes[0].set_title(f"Route demo reward curve, seed {ROUTE_DEMO_SEED} vs pass")
axes[0].legend()
axes[1].plot(common_steps, rebuilt_self.loc[common_steps, "money"] - base_self.loc[common_steps, "money"], color="#D97706", linewidth=2, label="paired money uplift")
axes[1].axhline(0, color="#111827", linewidth=1)
axes[1].set_ylabel("Uplift")
axes[1].legend()
axes[2].plot(common_steps, rebuilt_self.loc[common_steps, "battle_margin"], color="#0F766E", linewidth=2, label="rebuilt battle margin")
axes[2].axhline(0, color="#111827", linewidth=1)
axes[2].set_ylabel("Battle margin")
axes[2].set_xlabel("Step")
axes[2].legend()
for ax in axes:
    for step in range(72, 577, 72):
        ax.axvline(step, color="#CBD5E1", linewidth=0.8, alpha=0.7)
    ax.axvline(240, color="#B91C1C", linestyle="--", linewidth=1.4)
fig.tight_layout()
plt.show()

metric_rows = []
for row in special_rows:
    metrics = row["left_metrics"] or {}
    metric_rows.append({
        "match": f'{row["left"]} vs {row["right"]} / seed {row["seed"]}',
        "route": row["left_route"],
        "reward": row["left_reward"],
        "margin": row["left_margin"],
        "route_switches": metrics.get("route_switches", 0),
        "capacity_sales": metrics.get("capacity_sales", 0),
        "survival_overrides": metrics.get("survival_overrides", 0),
        "weed_blocks": metrics.get("weed_blocks", 0),
    })
print(pd.DataFrame(metric_rows).to_string(index=False))

## 7. Route Inventory Across 33 Tapes

v46d is not an online planner. It is an offline trajectory library with a runtime router:

- There are 8 shop types, and the first two draws use replacement, producing 64 ordered worlds.
- The archive contains 32 named pair tapes plus `default`, covering half of the possible pairs directly.
- An unmatched world is permanently routed to `default` at step 144.
- The rebuilt agent adds an alias for only one world, using a safe checkpoint validated on holdout simulations.

In [ ]:
TAPES = ORIGINAL._TAPES
DEFAULT = TAPES["default"]
SHOPS = [
    "BAKERY", "PIZZA_SHOP", "BRUNCH_SPOT", "YARN_STORE",
    "ICE_CREAM_SHOP", "PET_CAFE", "SMOOTHIE_SHOP", "FARMERS_MARKET",
]

def action_signature(action):
    return json.dumps(action, sort_keys=True, separators=(",", ":"))

def operation_counts(tape):
    counts = {
        "worker_actions": 0, "move": 0, "crop": 0, "animal": 0, "shed": 0,
        "market_orders": 0, "sell_units": 0, "hire": 0, "buy_land": 0,
    }
    for action in tape:
        workers = [action.get("farmer") or ["PASS"], *(action.get("hands") or [])]
        for worker in workers:
            op = worker[0] if worker else "PASS"
            if op != "PASS": counts["worker_actions"] += 1
            if op in {"NORTH", "SOUTH", "EAST", "WEST"}: counts["move"] += 1
            if op in {"PLANT", "WATER", "HARVEST", "FERTILIZE", "DIG"}: counts["crop"] += 1
            if op in {"BUILD_COOP", "BUILD_PASTURE", "FEED", "CARE", "COLLECT_FERTILIZER"}: counts["animal"] += 1
            if op in {"PICKUP", "DROP", "PLACE"}: counts["shed"] += 1
        for order in action.get("market") or []:
            counts["market_orders"] += 1
            if order and order[0] == "HIRE": counts["hire"] += 1
            if order and order[0] == "BUY_LAND": counts["buy_land"] += 1
            if order and order[0] == "SELL":
                try: counts["sell_units"] += int(order[2])
                except Exception: pass
    return counts

tape_rows = []
for route, tape in TAPES.items():
    signatures = [action_signature(a) for a in tape]
    default_signatures = [action_signature(a) for a in DEFAULT]
    different = [i for i, (a, b) in enumerate(zip(signatures, default_signatures)) if a != b]
    tape_rows.append({
        "route": route,
        "steps": len(tape),
        "diff_steps_vs_default": len(different),
        "first_diff": min(different) if different else None,
        "last_diff": max(different) if different else None,
        **operation_counts(tape),
    })
TAPE_PROFILE = pd.DataFrame(tape_rows).sort_values("diff_steps_vs_default", ascending=False)
print(f"routes={len(TAPES)}, named pairs={len(TAPES)-1}, tape length range={TAPE_PROFILE.steps.min()}..{TAPE_PROFILE.steps.max()}")
print("\nMost divergent routes:")
print(TAPE_PROFILE.head(12).to_string(index=False))

## 8. Route Coverage Matrix

Rows represent the first shop and columns represent the second shop. Dark cells have a dedicated tape; light cells currently fall back to `default`. Because shop sampling uses replacement, diagonal cells are also important.

In [ ]:
coverage = pd.DataFrame(0, index=SHOPS, columns=SHOPS, dtype=int)
for route in TAPES:
    if route == "default" or "__" not in route:
        continue
    first, second = route.split("__", 1)
    if first in coverage.index and second in coverage.columns:
        coverage.loc[first, second] = 1

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    coverage, annot=True, fmt="d", cmap=sns.color_palette(["#E2E8F0", "#0F766E"], as_cmap=True),
    cbar=False, linewidths=0.5, linecolor="white", square=True, ax=ax,
)
ax.set_title("Dedicated tape coverage for the first two shops")
ax.set_xlabel("Second shop")
ax.set_ylabel("First shop")
ax.tick_params(axis="x", rotation=50)
ax.tick_params(axis="y", rotation=0)
fig.tight_layout()
plt.show()
print(f"Dedicated coverage: {coverage.values.sum()}/64 = {coverage.values.mean():.1%}")

## 9. Route Divergence Curve and Checkpoints

`unique action signatures` measures how many distinct actions the 33 tapes contain at each step. It helps identify windows where routes genuinely begin to diverge:

- Low divergence indicates a lower checkpoint switching cost.
- High divergence means routes have already accumulated different inventory, positions, and production structures, making direct tape replacement likely to misalign state.
- A route must therefore not be replaced only because it has similar shop demand. State compatibility must also be checked.

In [ ]:
route_names = list(TAPES)
max_steps = min(len(TAPES[name]) for name in route_names)
diversity = []
for step in range(max_steps):
    diversity.append(len({action_signature(TAPES[name][step]) for name in route_names}))
DIVERSITY = pd.DataFrame({"step": np.arange(max_steps), "unique_actions": diversity})

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].plot(DIVERSITY.step, DIVERSITY.unique_actions, color="#0F766E", linewidth=1.8)
axes[0].set_title("Cross-route action diversity by step")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Unique action signatures")
for step in range(72, 577, 72):
    axes[0].axvline(step, color="#CBD5E1", linewidth=0.8)
axes[0].axvline(240, color="#B91C1C", linestyle="--", linewidth=1.3, label="validated switch")
axes[0].legend()

distance = TAPE_PROFILE[TAPE_PROFILE.route != "default"].nlargest(15, "diff_steps_vs_default").sort_values("diff_steps_vs_default")
axes[1].barh(distance.route, distance.diff_steps_vs_default, color="#D97706")
axes[1].set_title("Top route distances from default")
axes[1].set_xlabel("Different steps")
fig.tight_layout()
plt.show()

checkpoint_view = DIVERSITY[DIVERSITY.step.isin([72, 144, 216, 240, 288, 360, 432, 504, 576])]
print(checkpoint_view.to_string(index=False))

## 10. Multi-Route Action Density

Route names alone are insufficient. The following analysis compresses worker and market actions into daily densities to expose structural differences in production, movement, inventory handling, and capital expansion.

In [ ]:
def action_bucket_counts(action):
    buckets = {"move": 0, "crop": 0, "animal": 0, "shed": 0, "sell": 0, "capital": 0}
    workers = [action.get("farmer") or ["PASS"], *(action.get("hands") or [])]
    for worker in workers:
        op = worker[0] if worker else "PASS"
        if op in {"NORTH", "SOUTH", "EAST", "WEST"}: buckets["move"] += 1
        elif op in {"PLANT", "WATER", "HARVEST", "FERTILIZE", "DIG"}: buckets["crop"] += 1
        elif op in {"BUILD_COOP", "BUILD_PASTURE", "FEED", "CARE", "COLLECT_FERTILIZER"}: buckets["animal"] += 1
        elif op in {"PICKUP", "DROP", "PLACE"}: buckets["shed"] += 1
    for order in action.get("market") or []:
        if order and order[0] == "SELL": buckets["sell"] += 1
        elif order: buckets["capital"] += 1
    return buckets

selected_routes = ["default", "YARN_STORE__YARN_STORE", "YARN_STORE__SMOOTHIE_SHOP"]
density_rows = []
for route in selected_routes:
    for step, action in enumerate(TAPES[route]):
        density_rows.append({"route": route, "day": step // 24, **action_bucket_counts(action)})
density = pd.DataFrame(density_rows).groupby(["route", "day"], as_index=False).sum()

fig, axes = plt.subplots(len(selected_routes), 1, figsize=(12, 8.5), sharex=True)
palette = {"move": "#64748B", "crop": "#16A34A", "animal": "#7C3AED", "shed": "#0891B2", "sell": "#D97706", "capital": "#B91C1C"}
for ax, route in zip(axes, selected_routes):
    subset = density[density.route == route].set_index("day")
    for bucket, color in palette.items():
        ax.plot(subset.index, subset[bucket], label=bucket, color=color, linewidth=1.5)
    ax.set_title(route, loc="left", fontsize=10)
    ax.set_ylabel("actions/day")
axes[-1].set_xlabel("Day")
axes[0].legend(ncol=6, fontsize=8, loc="upper right")
fig.suptitle("Daily action density of representative tapes", y=1.01, fontweight="bold")
fig.tight_layout()
plt.show()

## 11. Replay Record Analysis: Event Tables and Opponent Route Features

The opponent's shed is hidden, so route inference should rely on **observable proxy variables**:

- Money, unlocked land, hired hands, and productive tiles.
- Crop and animal construction plus decreases in `yield_units`, which indicate harvest events.
- Sudden changes in market inventory and prices.
- Production-structure changes after shop unlocks.
- SELL timing, which is visible in downloaded replays but must be inferred indirectly from market changes during live play.

The event extractor below compresses a 720-step replay into a review-friendly record.

### Importing a Kaggle Replay JSON File

The function below converts a downloaded episode replay into the same long-form table. You can then reuse `replay_events`, `checkpoint_features`, and the curve-analysis cells directly.

In [ ]:
def normalize_replay_json(replay_or_path):
    if isinstance(replay_or_path, (str, Path)):
        replay = json.loads(Path(replay_or_path).read_text())
    else:
        replay = replay_or_path
    rows = []
    for frame_index, frame in enumerate(replay["steps"]):
        shared = frame[0]["observation"]
        farms = shared["farms"]
        shops = [str(x) for x in shared["town"].get("unlocked_shops", [])]
        prices = shared["market"].get("prices", {})
        inventory = shared["market"].get("inventory", {})
        for seat, state in enumerate(frame):
            obs = state["observation"]
            farm, opponent = farms[seat], farms[1 - seat]
            private = obs.get("private", {})
            shed = private.get("shed", {})
            own_tiles, opp_tiles = tile_features(farm), tile_features(opponent)
            row = {
                "frame": frame_index, "step": obs.get("step", frame_index),
                "day": obs.get("day", 0), "hour": obs.get("hour", 0), "seat": seat,
                "model": f"player_{seat}", "opponent": f"player_{1-seat}",
                "money": farm.get("money", 0), "opponent_money": opponent.get("money", 0),
                "battle_margin": farm.get("money", 0) - opponent.get("money", 0),
                "shed_total": sum(max(0, int(v or 0)) for v in shed.values()),
                "carried_total": sum(sum(max(0, int(v or 0)) for v in bag.values()) for bag in private.get("inventories", [])),
                "hands": len(farm.get("hands", [])), "opponent_hands": len(opponent.get("hands", [])),
                "land": len(farm.get("unlocked_quadrants", [])), "opponent_land": len(opponent.get("unlocked_quadrants", [])),
                "shops": "__".join(shops), **own_tiles,
                **{f"opponent_{k}": v for k, v in opp_tiles.items()},
                **action_features(state.get("action")),
            }
            for product in PRODUCTS:
                row[f"price_{product}"] = int(prices.get(product, 0) or 0)
                row[f"market_{product}"] = int(inventory.get(product, 0) or 0)
            rows.append(row)
    return pd.DataFrame(rows)

print("Usage: replay_df = normalize_replay_json('/path/to/replay.json')")
print("Then: replay_events(replay_df, seat=0), checkpoint_features(replay_df, observer_seat=0)")

## 12. Skill Rebuild Map: From Replays to Predictive and Reactive Defense

| Skill / harness | Inputs | Internal rollout | Permitted action | Current status |
|---|---|---|---|---|
| Opponent belief | Visible harvest or yield decrease and opponent position | Estimate carried cargo and shadow shed, then decay the estimate over time | Reorder capacity-SELL candidates only | Implemented and used conservatively |
| Capacity projection | Shed, inventories, worker actions, and market orders | Simulate the current turn and end-of-day storage in exact environment order | Sell only the minimum quantity needed to prevent overflow | Implemented; no negative result in the 64-seed stress set |
| Last-turn survival BT | Unwatered or unfed state and consecutive missed days | Determine whether the unit will die or become a weed at day end | Replace MOVE/PASS with WATER/FEED | Implemented as a low-frequency hard guard |
| Gated route checkpoint | First-two-shop world, step, and opponent activity | Compare a holdout-validated candidate tape | Switch only at step 240 when activity equals zero | Implemented for one world |
| Market counterfactual | Estimated opponent stock, town drain, and inventory/price curves | Forecast how sell timing changes marginal execution prices for both players | Delay, split, or reorder product sales | **Next priority** |
| Route posterior | Replay state signature and historical route library | Estimate `P(route | observations)` and run short-horizon rollouts over candidates | Checkpoint switch or local action replacement | **Next priority** |

### Mapping to *AI Agents in Depth*

- **Harness:** Deterministic constraints and observable metrics wrap a strong policy, making behavior reproducible and debuggable.
- **Model-based rollout:** Opponent belief and capacity projection are lightweight world models that do not depend solely on binary final rewards.
- **Fast and slow collaboration:** The tape is a low-latency fast policy; checkpoint route selection is low-frequency deliberation; the last-turn guard is a reactive controller.
- **Evaluation and reward-hacking controls:** BT ranking answers only relative strength. Paired uplift, p10, worst case, seat swaps, and no-regression gates constrain changes that raise the mean while damaging the tail.
- **Process rewards:** In addition to final money, track overflow units, survival losses, route switches, invalid actions, and inventory pressure to provide denser failure attribution.

## 13. Next-Round Growth Experiments, Ranked by Expected Return and Risk

1. **Counterfactual market defense**  
   Reconstruct town drain, visible harvests from both players, and market-inventory changes for each product. Forecast sell pressure over the next 1, 4, and 24 turns. Initially restrict the action space to splitting a current SELL, delaying it by one turn, or changing product order. Validate with paired seeds and seat swaps. Naive global front-running has already been falsified.

2. **Route posterior plus a safe checkpoint ensemble**  
   Record state signatures at steps 144, 216, 240, and 288: shop pair, money, land, hands, productive tiles, shed pressure, prices, and market-inventory displacement. Retrieve nearest historical replays, apply a compatibility filter to candidate routes, and then run bounded counterfactual rollouts. Every alias must satisfy `min uplift >= 0` or have an explicit strong-opponent gate.

3. **Replay-labelled advantage model**  
   Generate labels of the form `(state, candidate action, future 24/72-step money delta)`. Learn local action replacement before attempting to regenerate a 719-step plan. This preserves the strong tape baseline while gradually reducing brittle open-loop regions.

4. **State-aligned weed repair**  
   Replace fixed catch-up behavior with precondition tracking for worker position, inventory, target tile, and the next K planned steps. Return to the tape only when replay preconditions become valid again; otherwise enter a local task planner. Worst-case performance is critical because previous variants produced a -15,291 tail regression.

5. **Contextual BT and stratified leaderboards**  
   In addition to global BT, stratify by opponent family, shop world, seat, weed count, and shed pressure. Report rating intervals and identify the worlds in which gains occur. This prevents the global mean from hiding regressions in route subgroups.

In [ ]:
## 14. Review Checklist

- [ ] Does the formal comparison fix the same seed, opponent, and seat?
- [ ] Does it report reward uplift, margin uplift, p10, worst case, and positive/tie/negative counts?
- [ ] Does the BT input include both seats, a connected opponent graph, and consistent tie handling?
- [ ] Does every route alias have checkpoint compatibility checks and an opponent gate?
- [ ] Does replay logging capture shop unlocks, shed pressure, opponent activity, market inventory, and SELL events?
- [ ] Is every new defensive action restricted to the minimum necessary change?
- [ ] Are there independent holdouts for strong opponents, passive opponents, and mirrored self-play?
- [ ] Are runtime limits, exception fallback, and the submission rule of root-level `main.py` verified?

**Current recommendation:** Keep the rebuilt v46d agent as the stable baseline. Do not expand unscreened route aliases in the next version. Implement replay-verifiable market counterfactuals first, then use contextual BT and paired uplift to decide whether the change should enter the submission.